# Transferarbeit Data Science – Credit Card Fraud Detection

**Modul:** Data Science · TEKO Luzern  
**Gruppe:** Daniel Halter, [Nachname2 Vorname2]  
**Datensatz:** D – Credit Card Fraud Detection (Kaggle, ULB)  
**Datum:** September 2026

---
> **Hinweis zum Aufbau:** Dieses Notebook ist die *gemeinsame Basis* (Teil 1 + Teil 2).
> Die beiden Modelle in Teil 3/4 werden auf getrennten Branches gebaut
> (`feature/model-logreg`, `feature/model-rf`) und am Schluss in `main` zusammengeführt.
> **Kontrakt zwischen den Branches:** Beide Modelle nutzen dieselben Variablen
> `X_train_scaled, X_test_scaled, y_train, y_test` und dieselbe Funktion `evaluate_model(...)`.
> So ist der Vergleich am Ende garantiert fair (gleiche Daten, gleiche Metriken).

## Teil 1 – Use-Case-Analyse

> *Entwurf – bitte vor Abgabe in eigenen Worten prüfen und v.a. die Reflexion (5.) ergänzen.
> Die eigenständige Reflexion zu Risiken bringt in der Bewertung die volle Punktzahl.*

**1. Datensatz-Beschreibung – Was wurde erfasst, von wem und warum?**  
Der Datensatz enthält Kreditkartentransaktionen europäischer Karteninhaber aus dem September 2013,
erhoben und veröffentlicht von der Forschungsgruppe *Worldline & ULB (Machine Learning Group,
Université Libre de Bruxelles)*. Erfasst wurden 284'807 Transaktionen über zwei Tage, davon nur
492 Betrugsfälle (~0.172 %). Ziel der Erhebung war die Entwicklung von Modellen zur automatischen
Betrugserkennung. Aus Datenschutzgründen sind die 28 Merkmale `V1`–`V28` das Ergebnis einer
PCA-Transformation und daher inhaltlich nicht interpretierbar; im Klartext vorhanden sind nur
`Time` (Sekunden seit der ersten Transaktion) und `Amount` (Transaktionsbetrag).

**2. Business-Fragestellung (ein Satz):**  
*Kann anhand der Transaktionsmerkmale einer Kreditkartenzahlung vorhergesagt werden, ob es sich
um Betrug handelt?*

**3. Art des Problems:**  
Es handelt sich um eine **binäre Klassifikation** (Zielvariable `Class`: 0 = legitim, 1 = Betrug).
Die Zielvariable ist kategorisch und beschriftet (supervised), daher weder Regression (keine
kontinuierliche Zielgrösse) noch Clustering (Labels sind vorhanden).

**4. Mehrwert für ein Unternehmen:**  
Eine automatische Erkennung erlaubt es, verdächtige Transaktionen in Echtzeit zu blockieren oder zur
manuellen Prüfung zu markieren, bevor ein finanzieller Schaden entsteht. Das senkt direkte
Betrugskosten und Chargeback-Gebühren und schützt das Vertrauen der Kundschaft – bei gleichzeitig
geringerer manueller Prüflast.

**5. Einschränkungen, Risiken, ethische Fragen:**  
- **Extreme Klassenungleichheit (0.172 %):** Accuracy ist als Metrik wertlos – ein Modell, das immer
  „legitim" sagt, erreicht 99.8 % und erkennt keinen einzigen Betrug. Fokus daher auf Recall/Precision.
- **Fehler-Trade-off:** Ein *False Negative* (übersehener Betrug) kostet Geld; ein *False Positive*
  (fälschlich blockierte echte Zahlung) verärgert Kundschaft. Die Gewichtung ist eine
  Geschäftsentscheidung, keine rein technische.
- **Black-Box-Merkmale:** `V1`–`V28` sind anonymisiert – Entscheidungen sind schwer erklärbar, was
  bei einer Ablehnung gegenüber Kundschaft/Regulierung problematisch sein kann.
- *(hier eigene Gedanken ergänzen: Datenalter/Übertragbarkeit, Bias, DSGVO …)*

## Teil 2 – Datenprozessierung

### 2.1 Import und erster Überblick
Wir laden den Datensatz über einen **relativen Pfad** (`data/creditcard.csv`), damit das Notebook
auf beiden Geräten und beim Dozenten identisch läuft.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Zentrale Konstante -> beide Branches ziehen denselben Split
RANDOM_STATE = 42

df = pd.read_csv("data/creditcard.csv")
print("Form (Zeilen, Spalten):", df.shape)

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df.describe()

**Beobachtungen:** 284'807 Transaktionen, 31 Spalten, alle numerisch (kein Encoding nötig).
`Time` und `Amount` liegen in Klartext-Skalen vor, `V1`–`V28` sind bereits standardisiert
(Mittelwert ≈ 0, ähnliche Streuung – Ergebnis der PCA). `Amount` ist stark rechtsschief
(Median ≪ Maximum). Die Zielvariable `Class` ist massiv unausgeglichen (siehe 2.3).

### 2.2 Datenqualität prüfen und bereinigen

In [ ]:
# Fehlende Werte
missing = df.isnull().sum()
print("Fehlende Werte total:", int(missing.sum()))
print(missing[missing > 0] if missing.sum() else "-> Keine fehlenden Werte vorhanden.")

**Fehlende Werte:** Der Datensatz ist vollständig – keine Imputation oder Zeilenentfernung nötig.

In [ ]:
# Dubletten
n_dup = int(df.duplicated().sum())
print(f"Exakte Dubletten: {n_dup}")
df = df.drop_duplicates().reset_index(drop=True)
print("Form nach Entfernen der Dubletten:", df.shape)

**Dubletten:** Es existieren einige exakt identische Zeilen. Da eine reale Transaktion durch
Zeitstempel/Betrag/PCA-Merkmale praktisch eindeutig ist, werten wir exakte Duplikate als
technische Doppelerfassung und entfernen sie, um Verzerrungen und Data Leakage über den
Train/Test-Split hinweg zu vermeiden.

In [ ]:
# Ausreisser-Analyse (IQR-Methode) am Beispiel 'Amount' -- nur Analyse, keine Entfernung
Q1, Q3 = df["Amount"].quantile(0.25), df["Amount"].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
n_out = int(((df["Amount"] < lower) | (df["Amount"] > upper)).sum())
print(f"IQR-Ausreisser in 'Amount': {n_out} ({n_out/len(df)*100:.1f} %)")

**Ausreisser – Entscheidung: bewusst behalten.** Bei einem Fraud-Datensatz ist der Betrug
*selbst* das seltene, extreme Ereignis. Würden wir statistische Ausreisser (z. B. sehr hohe Beträge)
entfernen, löschten wir genau das Signal, das das Modell lernen soll. Ausreisserentfernung wäre hier
kontraproduktiv – wir dokumentieren die Analyse, verändern die Daten aber nicht.

### 2.3 Explorative Datenanalyse (EDA)

In [ ]:
# Plot 1 -- Verteilung der Zielvariable (linear + logarithmisch, da sonst der Betrugsbalken unsichtbar ist)
counts = df["Class"].value_counts().sort_index()
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
labels = ["Legitim (0)", "Betrug (1)"]
ax[0].bar(labels, counts.values, color=["#4c72b0", "#c44e52"])
ax[0].set_title("Klassenverteilung (linear)"); ax[0].set_ylabel("Anzahl Transaktionen")
ax[1].bar(labels, counts.values, color=["#4c72b0", "#c44e52"])
ax[1].set_yscale("log"); ax[1].set_title("Klassenverteilung (logarithmisch)")
ax[1].set_ylabel("Anzahl (log-Skala)")
fig.suptitle("Verteilung der Zielvariable 'Class'")
plt.tight_layout(); plt.show()
print(f"Betrugsanteil: {counts[1]/counts.sum()*100:.3f} %  ({counts[1]} von {counts.sum()})")

**Interpretation:** Nur ~0.17 % aller Transaktionen sind Betrug. Diese extreme Ungleichheit ist
der zentrale Befund der ganzen Arbeit: Sie zwingt uns zu (a) einem *stratifizierten* Split, (b) dem
Verzicht auf Accuracy als Leitmetrik und (c) Massnahmen gegen die Imbalance (`class_weight='balanced'`).

In [ ]:
# Plot 2 -- Betragsverteilung getrennt nach Klasse (frei gewaehlte Darstellung)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(df.loc[df.Class == 0, "Amount"], bins=50, color="#4c72b0")
ax[0].set_title("Betrag – Legitim"); ax[0].set_xlabel("Amount"); ax[0].set_ylabel("Anzahl (log)")
ax[0].set_yscale("log")
ax[1].hist(df.loc[df.Class == 1, "Amount"], bins=50, color="#c44e52")
ax[1].set_title("Betrag – Betrug"); ax[1].set_xlabel("Amount"); ax[1].set_ylabel("Anzahl")
fig.suptitle("Verteilung von 'Amount' nach Klasse")
plt.tight_layout(); plt.show()
print(df.groupby("Class")["Amount"].describe()[["mean", "50%", "max"]])

**Interpretation:** Beide Klassen sind rechtsschief. Betrugstransaktionen häufen sich tendenziell
bei kleineren Beträgen, während Legitim-Transaktionen die ganze Spanne abdecken. Der Betrag allein
trennt die Klassen nicht sauber – das begründet, warum wir die (informativeren) PCA-Merkmale brauchen.

In [ ]:
# Plot 3 -- Korrelation der Merkmale mit der Zielvariable
corr_target = df.corr(numeric_only=True)["Class"].drop("Class").sort_values()
plt.figure(figsize=(10, 8))
colors = ["#c44e52" if v < 0 else "#4c72b0" for v in corr_target.values]
plt.barh(corr_target.index, corr_target.values, color=colors)
plt.title("Korrelation jedes Merkmals mit 'Class'")
plt.xlabel("Pearson-Korrelation"); plt.axvline(0, color="black", linewidth=0.8)
plt.tight_layout(); plt.show()

**Interpretation:** Einige PCA-Merkmale (z. B. V14, V12, V10, V17) zeigen eine deutliche negative
Korrelation mit Betrug, andere eine positive. Trotz Anonymisierung sind also klare Zusammenhänge
vorhanden – ein gutes Zeichen dafür, dass die Modelle Struktur lernen können. `Amount`/`Time`
korrelieren kaum direkt mit `Class`.

**Zusammenfassung EDA:** (1) Extreme Imbalance ~0.17 % bestimmt die gesamte Methodik.
(2) `Amount` ist rechtsschief und allein wenig trennscharf → Log-Transformation sinnvoll.
(3) Mehrere V-Merkmale sind klar mit Betrug assoziiert → gute Voraussetzung fürs Modelling.

### 2.4 Feature Engineering
Da `V1`–`V28` bereits PCA-transformiert und nicht interpretierbar sind, setzen wir am einzigen
Klartext-Material an: `Time` und `Amount`.

In [ ]:
# Feature 1: Tageszeit (Stunde) aus 'Time'. 'Time' = Sekunden seit erster Transaktion.
df["Hour"] = (df["Time"] // 3600) % 24

# Feature 2: Log-Transformation von 'Amount' gegen die starke Rechtsschiefe.
df["Amount_log"] = np.log1p(df["Amount"])

df[["Time", "Hour", "Amount", "Amount_log"]].head()

**Begründung der Features:**
- **`Hour`** – Betrug folgt oft Tageszeit-Mustern (z. B. nachts). Die absolute Sekundenzahl `Time`
  ist als Rohwert wenig aussagekräftig; die Stunde des Tages ist ein interpretierbares, zyklisches
  Merkmal.
- **`Amount_log`** – `log1p` staucht die stark rechtsschiefe Betragsverteilung und verhindert, dass
  wenige Grossbeträge das Modell (v. a. die Logistic Regression) dominieren.

Für das Modelling verwenden wir `Hour` und `Amount_log` und lassen die Rohspalten `Time`/`Amount`
weg, um Redundanz zu vermeiden.

### 2.5 Train/Test-Split
Aufteilung 80/20 mit **Stratifizierung** und festem `random_state`. Beides ist hier entscheidend.

In [ ]:
feature_cols = [c for c in df.columns if c not in ["Class", "Time", "Amount"]]
X = df[feature_cols]
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

# Skalierung: Scaler NUR auf Trainingsdaten fitten (kein Leakage), dann beide transformieren.
# V1-V28 sind bereits standardisiert -> nur die neuen Features skalieren.
cols_to_scale = ["Amount_log", "Hour"]
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test_scaled[cols_to_scale]  = scaler.transform(X_test[cols_to_scale])

print("Train:", X_train_scaled.shape, "| Test:", X_test_scaled.shape)
print(f"Betrugsanteil Train: {y_train.mean()*100:.3f} %")
print(f"Betrugsanteil Test:  {y_test.mean()*100:.3f} %")

**Begründung:**
- **Stratifizierung (`stratify=y`)** ist bei 0.17 % Betrug Pflicht: Ohne sie könnten im Testset
  zufällig zu wenige Betrugsfälle landen und die Metriken würden unbrauchbar. Die Ausgabe bestätigt
  den identischen Betrugsanteil in Train und Test.
- **Fester `random_state=42`** macht den Split reproduzierbar – beide Gruppenmitglieder trainieren
  dadurch auf **exakt denselben Daten**, was den Modellvergleich erst fair macht.
- **Scaler nur auf Train gefittet** – sonst würde Information aus dem Testset ins Training „leaken".

### Gemeinsame Evaluations-Funktion (Basis – von beiden Branches genutzt)
Beide Modelle rufen dieselbe Funktion auf. Das garantiert identische Metriken und macht den
Vergleich in Teil 4 zu einem Zweizeiler.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay)

def evaluate_model(name, model, X_te, y_te):
    """Berechnet die Standard-Klassifikationsmetriken, zeigt die Confusion Matrix
    und gibt ein Ergebnis-Dict mit einheitlichen Keys zurueck."""
    y_pred = model.predict(X_te)
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_te)[:, 1]
    else:
        y_score = model.decision_function(X_te)

    results = {
        "Modell":    name,
        "Accuracy":  accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred, zero_division=0),
        "Recall":    recall_score(y_te, y_pred, zero_division=0),
        "F1":        f1_score(y_te, y_pred, zero_division=0),
        "ROC-AUC":   roc_auc_score(y_te, y_score),
    }

    cm = confusion_matrix(y_te, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["Legitim", "Betrug"]).plot(cmap="Blues")
    plt.title(f"Confusion Matrix – {name}")
    plt.show()
    return results

---
## ⬇️ AB HIER BRANCH-ARBEIT ⬇️
Alles oberhalb ist die stabile gemeinsame Basis auf `main`.
Ab hier gilt die Aufteilung:
- **`feature/model-logreg`** → Abschnitt „Modell A" (Daniel)
- **`feature/model-rf`** → Abschnitt „Modell B" (Partner)

Am Schluss beide Branches nach `main` mergen, dann Teil 4 (Vergleich) füllen und einmal
`Kernel → Restart & Run All` ausführen.


## Teil 3 – Modellwahl und Training

### Modell A – Logistic Regression  · Branch `feature/model-logreg`
**Funktionsweise (2–3 Sätze):** Die logistische Regression modelliert die Wahrscheinlichkeit der
Klasse „Betrug" als logistische Funktion einer gewichteten Summe der Merkmale. Sie ist ein
lineares, gut interpretierbares Basismodell (Koeffizienten = Merkmalseinfluss) und dient hier als
schnelle, transparente Baseline. `class_weight='balanced'` gewichtet die seltene Betrugsklasse
stärker, um der Imbalance entgegenzuwirken.

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(
    class_weight="balanced",     # gegen die Imbalance
    max_iter=1000,
    random_state=RANDOM_STATE,
)
logreg.fit(X_train_scaled, y_train)

results_logreg = evaluate_model("Logistic Regression", logreg, X_test_scaled, y_test)
results_logreg

**TODO (Branch-Arbeit Daniel):**
- Vorhersagen übersichtlich ausgeben (Teil 3.3).
- Mind. 1 Hyperparameter dokumentiert variieren (z. B. `C`, `solver`) und Effekt beschreiben (Teil 3.2 Tuning).
- Kurz-Interpretation der Confusion Matrix in eigenen Worten.

### Modell B – Random Forest  · Branch `feature/model-rf`
**Funktionsweise (2–3 Sätze):** Ein Random Forest ist ein Ensemble vieler Entscheidungsbäume, die
je auf zufälligen Daten- und Merkmalsstichproben trainieren; die Vorhersage ist die Mehrheit ihrer
Stimmen. Das Modell erfasst nichtlineare Zusammenhänge, ist robust gegen Ausreisser und liefert
eine Feature Importance. `class_weight='balanced'` adressiert wiederum die Imbalance.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf.fit(X_train_scaled, y_train)

results_rf = evaluate_model("Random Forest", rf, X_test_scaled, y_test)
results_rf

In [ ]:
# Feature Importance (Random Forest)
importances = (pd.Series(rf.feature_importances_, index=X_train_scaled.columns)
                 .sort_values(ascending=False).head(10))
plt.figure(figsize=(8, 5))
importances[::-1].plot(kind="barh", color="#4c72b0")
plt.title("Top-10 Feature Importance – Random Forest")
plt.xlabel("Wichtigkeit"); plt.tight_layout(); plt.show()

**TODO (Branch-Arbeit Partner):**
- Vorhersagen übersichtlich ausgeben (Teil 3.3).
- Mind. 1 Hyperparameter dokumentiert variieren (z. B. `n_estimators`, `max_depth`) und Effekt beschreiben.
- Kurz-Interpretation der Feature Importance in eigenen Worten.

## Teil 4 – Auswertung, Fazit und Verbesserungsmöglichkeiten
> *In `main` nach dem Merge beider Branches füllen.*

### 4.1 Modell-Evaluation – Vergleichstabelle

In [ ]:
vergleich = pd.DataFrame([results_logreg, results_rf]).set_index("Modell")
vergleich.round(4)

### 4.2 Interpretation
> *In eigenen Worten ausfüllen – hier liegen die Punkte.*
- Warum ist **Accuracy** hier irreführend, und welche Metrik ist für Fraud am wichtigsten
  (Recall = Anteil erkannter Betrugsfälle) und warum?
- Welches Modell schneidet besser ab – und ist der Unterschied den Aufwand/Verlust an
  Interpretierbarkeit wert?
- Recall-vs-Precision-Trade-off im Geschäftskontext einordnen.
- Wichtigste Features (aus RF Feature Importance).

### 4.3 Fazit
> Business-Frage aus Teil 1 klar beantworten: Lässt sich Betrug anhand der Merkmale vorhersagen?
> Wurde das Ziel erreicht?

### 4.4 Verbesserungsmöglichkeiten (mind. 3, konkret + begründet)
1. **Resampling (SMOTE / Undersampling)** via `imbalanced-learn` statt/zusätzlich zu `class_weight`
   und Vergleich der Wirkung.
2. **Threshold-Tuning** – Entscheidungsschwelle bewusst vom Default 0.5 verschieben, um Recall zu
   erhöhen (mehr Betrug erkennen), inkl. Precision-Recall-Kurve.
3. **Stärkeres Modell / Tuning** – XGBoost mit `scale_pos_weight` und systematischem Grid Search.
4. *(eigene Idee: mehr/aktuellere Daten, weitere zeitbasierte Features, Kostenmatrix …)*

## Anhang – Einsatz von KI-Tools
> *Pflichtabschnitt gemäss Aufgabenstellung – ehrlich und konkret ausfüllen.*

Für diese Arbeit wurde **[Claude / ChatGPT / …]** eingesetzt für:
- Aufbau des Notebook-Gerüsts und der Repo-Struktur (Git-Branch-Workflow, `.gitignore`, `requirements.txt`).
- Erstellung und Erläuterung des Basis-Codes für Datenimport, Bereinigung, EDA und Train/Test-Split.
- *(ergänzen, wofür ihr KI beim Modelling / bei Formulierungen konkret genutzt habt)*

Alle Ergebnisse wurden von uns geprüft, angepasst und im Notebook eigenständig interpretiert.

**Quellen:** Datensatz – Kaggle „Credit Card Fraud Detection" (ULB Machine Learning Group).